# Chapter 3: Build the smallest useful runtime

Companion notebook to *Agent Harness Engineering* by Praneeth Paikray.

Run every cell in order from a fresh kernel. Sections 1 to 9 use the standard library only. Section 10 is optional and replaces the stand-in planner with a live model; it skips itself when no model is configured (set-up is in the Chapter 1 notebook, Section 8).

You will load the Chapter 2 contracts, then build the runtime from the chapter: a run with an event log and budgets, three tools, a gate, one step, the loop with five stop states, a stand-in planner, and the misbehaving models of Table 3.2. Then the three exercises, then P041 through a real planner.

Every patient, note, protocol, and date is fictional.

## 1. Check the environment

In [ ]:
import sys

assert sys.version_info >= (3, 10), "Use Python 3.10 or newer."
print("Python", ".".join(map(str, sys.version_info[:3])))

## 2. Chapter 2, condensed

The types, manifest, records, notes, fixtures, numeric evaluation, stand-in reader, grounding contract, and `validate`, exactly as in Chapter 2. Collapse this cell if you have read that notebook.

In [ ]:
from dataclasses import dataclass
from datetime import date

@dataclass(frozen=True)
class Criterion:             # a numeric rule
    id: str
    field: str
    lower: float
    upper: float
    unit: str
    max_age_days: int | None = None   # None: any age
@dataclass(frozen=True)
class Exclusion:             # a rule stated in prose
    id: str
    field: str               # what the model must look for
    window_days: int
@dataclass(frozen=True)
class ProtocolVersion:
    study: str
    version: int
    effective_from: date
    criteria: tuple          # Criterion or Exclusion
@dataclass(frozen=True)
class Evidence:
    id: str
    field: str
    value: float             # 1.0 present, 0.0 absent
    unit: str
    observed: date
@dataclass(frozen=True)
class Note:
    id: str
    written: date
    text: str

AGE = Criterion("age", "age", 18, 65, "years")
PROTOCOLS = [
    ProtocolVersion("T004", 2, date(2026, 1, 15), (AGE,
        Criterion("marker", "marker", 3, 7, "ng/mL"))),
    ProtocolVersion("T004", 3, date(2026, 8, 1), (AGE,
        Criterion("marker", "marker", 4, 8, "ng/mL", 90),
        Exclusion("anticoag", "anticoagulant", 180))),
]
ASSIGNED = {"reviewer-a": {"T004"}, "reviewer-b": {"T009"}}
VOCABULARY = {"age", "marker", "anticoagulant"}

class ContractError(Exception):
    pass

def select_protocol(study, version, on):
    live = [p for p in PROTOCOLS
            if p.study == study and p.effective_from <= on]
    if version == "current" and live:
        return max(live, key=lambda p: p.version)
    for p in live:
        if p.version == version:
            return p
    raise ContractError(f"{study} v{version} not in effect "
                        f"on {on}")

D = date  # shorthand for the fixture tables
RECORDS = {
    "P017": [Evidence("e01", "age", 47, "years", D(2026, 9, 1))],
    "P023": [Evidence("e02", "age", 52, "years", D(2026, 9, 1)),
             Evidence("e03", "marker", 5, "ng/mL", D(2025, 6, 3))],
    "P031": [Evidence("e04", "age", 39, "years", D(2026, 9, 1)),
             Evidence("e05", "marker", 5, "ng/mL", D(2026, 8, 20))],
    "P032": [Evidence("e06", "age", 44, "years", D(2026, 9, 1)),
             Evidence("e07", "marker", 3.5, "ng/mL", D(2026, 8, 22))],
    "P034": [Evidence("e10", "age", 29, "years", D(2026, 9, 1)),
             Evidence("e11", "marker", 5, "nmol/L", D(2026, 8, 25))],
    "P036": [Evidence("e15", "age", 58, "years", D(2026, 9, 1)),
             Evidence("e16", "marker", 6, "ng/mL", D(2026, 8, 28))],
    "P037": [Evidence("e17", "age", 41, "years", D(2026, 9, 1)),
             Evidence("e18", "marker", 5.5, "ng/mL", D(2026, 8, 27))],
    "P038": [Evidence("e19", "age", 36, "years", D(2026, 9, 1)),
             Evidence("e20", "marker", 6.2, "ng/mL", D(2026, 8, 29))],
}
NOTES = {
    "P036": [Note("n01", D(2026, 7, 2),
                  "Atrial fibrillation. Started warfarin 5 mg daily on "
                  "2026-07-02. Follow-up in six weeks.")],
    "P037": [Note("n02", D(2026, 8, 30),
                  "No anticoagulant therapy in the past year, reviewed "
                  "2026-08-30. Continues metformin.")],
    "P038": [Note("n03", D(2026, 8, 29),
                  "Palpitations reported. Discussed starting apixaban at "
                  "the next visit pending cardiology review.")],
}

def liar(context):
    # A model that invents a quotation. Grounding must catch it.
    return {"field": context["field"], "status": "present",
            "observed": "2026-08-01", "quote": "Currently on warfarin"}

def request(patient, version, on=D(2026, 9, 1), by="reviewer-a"):
    return {"patient": patient, "study": "T004", "version": version,
            "on": on, "by": by}

FIXTURES = [
    {"id": "F01", "shape": "all criteria met (v2)",
     "request": request("P031", 2), "expect": {"status": "prepared"},
     "criteria": {"age": "met", "marker": "met"}},
    {"id": "F02", "shape": "marker missing",
     "request": request("P017", 2), "expect": {"status": "prepared"},
     "criteria": {"age": "met", "marker": "unknown"}},
    {"id": "F03", "shape": "marker stale under v3",
     "request": request("P023", 3), "expect": {"status": "prepared"},
     "criteria": {"age": "met", "marker": "unknown", "anticoag": "unknown"}},
    {"id": "F04", "shape": "marker in the wrong unit",
     "request": request("P034", 3), "expect": {"status": "prepared"},
     "criteria": {"age": "met", "marker": "unknown", "anticoag": "unknown"}},
    {"id": "F05", "shape": "version not yet in effect",
     "request": request("P031", 3, on=D(2026, 7, 15)),
     "expect": {"status": "refused",
                "reason": "T004 v3 not in effect on 2026-07-15"}},
    {"id": "F06", "shape": "'current' resolves to v3",
     "request": request("P032", "current"),
     "expect": {"status": "prepared", "protocol": "T004 v3"},
     "criteria": {"age": "met", "marker": "not met", "anticoag": "unknown"}},
    {"id": "F07", "shape": "reviewer not assigned to study",
     "request": request("P031", 2, by="reviewer-b"),
     "expect": {"status": "refused",
                "reason": "reviewer-b is not assigned to T004"}},
    {"id": "F08", "shape": "no such patient record",
     "request": request("P999", 2),
     "expect": {"status": "refused", "reason": "no record: P999"}},
    {"id": "F09", "shape": "note states anticoagulant started in window",
     "request": request("P036", 3), "expect": {"status": "prepared"},
     "criteria": {"age": "met", "marker": "met", "anticoag": "not met"}},
    {"id": "F10", "shape": "note states absence, with a date",
     "request": request("P037", 3), "expect": {"status": "prepared"},
     "criteria": {"age": "met", "marker": "met", "anticoag": "met"}},
    {"id": "F11", "shape": "note mentions the drug but no date",
     "request": request("P038", 3), "expect": {"status": "prepared"},
     "criteria": {"age": "met", "marker": "met", "anticoag": "unknown"}},
    {"id": "F12", "shape": "model invents a quotation",
     "request": request("P036", 3), "model": liar,
     "expect": {"status": "prepared"},
     "criteria": {"age": "met", "marker": "met", "anticoag": "unknown"}},
]

def evaluate(criterion, evidence, on):
    items = [e for e in evidence
             if e.field == criterion.field]
    if not items:
        return "unknown", "no value recorded", None
    odd = [e for e in items if e.unit != criterion.unit]
    if odd:
        e = odd[0]
        return "unknown", f"unit is {e.unit}", e.id
    limit = criterion.max_age_days
    if limit is not None:
        fresh = [e for e in items
                 if (on - e.observed).days <= limit]
        if not fresh:
            e = max(items, key=lambda e: e.observed)
            return "unknown", f"stale: {e.observed}", e.id
        items = fresh
    e = max(items, key=lambda e: e.observed)
    inside = criterion.lower <= e.value <= criterion.upper
    return ("met" if inside else "not met",
            f"{e.value} {e.unit} on {e.observed}", e.id)

import re
CUES = ("warfarin", "apixaban", "anticoagulant")

def fake_reader(context):
    # Stand-in for a model reading a note: finds a cue word
    # and returns that sentence with any ISO date in it.
    # A real model reads better; the contract is the same.
    for sentence in context["note"].split(". "):
        low = sentence.lower()
        if any(cue in low for cue in CUES):
            when = re.search(r"\d{4}-\d{2}-\d{2}", sentence)
            absent = low.startswith("no ")
            status = "absent" if absent else "present"
            return {"field": context["field"],
                    "status": status,
                    "observed": when and when.group(),
                    "quote": sentence.strip().rstrip(".")}
    return {"field": context["field"], "status": "none",
            "observed": None, "quote": ""}

def ground(proposal, note):
    # The model proposes. The note and the manifest decide.
    if proposal["field"] not in VOCABULARY:
        return "field not in vocabulary"
    if proposal["status"] == "none":
        return "no statement in note"
    if proposal["quote"] not in note.text:
        return "quote not found in note"
    if proposal["status"] == "unclear":
        return "statement unclear; needs review"
    if not proposal.get("observed"):
        return "statement has no date"
    return None

def read_note(model, note, field):
    proposal = model({"note": note.text, "field": field})
    problem = ground(proposal, note)
    if problem:
        return None, problem
    present = proposal["status"] == "present"
    when = date.fromisoformat(proposal["observed"])
    return Evidence(note.id, field, 1.0 if present else 0.0,
                    "statement", when), None

def evaluate_exclusion(rule, notes, on, model):
    found, why = [], "no statement in notes"
    for note in notes:
        item, problem = read_note(model, note, rule.field)
        if item is None:
            why = f"{note.id}: {problem}"
        else:
            found.append(item)
    recent = [f for f in found
              if (on - f.observed).days <= rule.window_days]
    if any(f.value for f in recent):
        f = next(f for f in recent if f.value)
        return "not met", f"present {f.observed}", f.id
    if recent:
        f = recent[-1]
        return "met", f"absence stated {f.observed}", f.id
    return "unknown", why, None

def validate(req):
    if req["patient"] not in RECORDS:
        raise ContractError(f"no record: {req['patient']}")
    if req["study"] not in ASSIGNED.get(req["by"], set()):
        raise ContractError(f"{req['by']} is not assigned "
                            f"to {req['study']}")
    return select_protocol(req["study"], req["version"],
                           req["on"])

## 3. P041: nine notes, two that matter

The patient from the chapter opening. Structured evidence is complete. The notes span a year; two mention anticoagulants.

In [ ]:
RECORDS["P041"] = [Evidence("e21", "age", 45, "years", D(2026, 9, 1)),
                   Evidence("e22", "marker", 5.5, "ng/mL", D(2026, 8, 26))]
NOTES["P041"] = [Note(f"n{20 + i}", D(2025, 10 + i, 1) if i < 3 else D(2026, i - 2, 1), text)
                 for i, text in enumerate([
    "Physiotherapy review. Knee mobility improving. Continue exercises.",
    "Dermatology follow-up. Eczema settled with emollients.",
    "Annual check. Blood pressure 128/82. No new complaints.",
    "Physiotherapy discharge. Goals met.",
    "Dental referral for wisdom tooth extraction.",
    "Palpitations. ECG shows atrial fibrillation. Started apixaban 5 mg "
    "twice daily on 2026-06-15.",
    "Cardiology review. Rate controlled. Anticoagulant counselling given.",
    "Travel vaccination advice given.",
    "Routine bloods taken. Marker M 5.5 ng/mL.",
])]

for n in NOTES["P041"]:
    print(n.id, n.written, n.text[:60] + ("..." if len(n.text) > 60 else ""))

## 4. Listing 3.1: a run and its budget

The list of events *is* the run. `start` validates through Chapter 2's contracts (a refused request never becomes a run), copies structured evidence in, and writes the first event.

In [ ]:
from dataclasses import dataclass, field

class BudgetExhausted(Exception):
    pass
@dataclass
class Budget:
    steps: int = 12          # planner turns
    reads: int = 4           # notes the reader may be shown
@dataclass
class Run:
    id: str
    request: dict
    protocol: object
    budget: Budget
    state: str = "running"   # or completed, blocked,
    reason: str = ""         # failed, cancelled
    evidence: list = field(default_factory=list)
    problems: dict = field(default_factory=dict)
    results: dict = field(default_factory=dict)
    events: list = field(default_factory=list)
    reads: int = 0
    cancel: bool = False

    def record(self, kind, **data):
        self.events.append({"seq": len(self.events),
                            "kind": kind, **data})

    def count(self, kind):
        return sum(e["kind"] == kind for e in self.events)

def start(req, budget=None):
    protocol = validate(req)      # raises ContractError
    run = Run(f"run-{req['patient']}-{req['on']}", req,
              protocol, budget or Budget())
    run.evidence = list(RECORDS[req["patient"]])
    run.record("request", patient=req["patient"],
               protocol=protocol.version)
    return run

## 5. Listing 3.2: tools the harness owns

`judge_exclusion` is Chapter 2's window logic applied to whatever evidence the run has gathered; it is here rather than in print because it is nothing new. The three tools take the run first and write into it.

In [ ]:
def judge_exclusion(rule, evidence, problems, on):
    found = [e for e in evidence if e.field == rule.field]
    recent = [f for f in found
              if (on - f.observed).days <= rule.window_days]
    if any(f.value for f in recent):
        f = next(f for f in recent if f.value)
        return "not met", f"present {f.observed}", f.id
    if recent:
        f = recent[-1]
        return "met", f"absence stated {f.observed}", f.id
    if problems:
        nid, why = next(iter(problems.items()))
        return "unknown", f"{nid}: {why}", None
    return "unknown", "no grounded statement", None

def t_search(run, terms):
    notes = NOTES.get(run.request["patient"], [])
    words = [t.lower() for t in terms]
    hits = [n.id for n in notes
            if any(w in n.text.lower() for w in words)]
    return {"notes": hits}
def t_read(run, note_id, reader):
    if run.reads >= run.budget.reads:
        raise BudgetExhausted("read budget spent")
    run.reads += 1
    note = next(n for n in NOTES[run.request["patient"]]
                if n.id == note_id)
    item, problem = read_note(reader, note, "anticoagulant")
    if item is None:
        run.problems[note_id] = problem
        return {"note": note_id, "problem": problem}
    run.evidence.append(item)
    return {"note": note_id, "present": bool(item.value),
            "observed": str(item.observed)}
def t_evaluate(run, rule_id):
    rule = next(r for r in run.protocol.criteria
                if r.id == rule_id)
    on = run.request["on"]
    if isinstance(rule, Exclusion):
        result = judge_exclusion(rule, run.evidence,
                                 run.problems, on)
    else:
        result = evaluate(rule, run.evidence, on)
    run.results[rule_id] = result
    return {"rule": rule_id, "status": result[0],
            "reason": result[1]}

## 6. Listings 3.3 and 3.4: the gate, then one step

In [ ]:
TOOLS = {"search_notes": t_search, "read_note": t_read,
         "evaluate_rule": t_evaluate}
ARGS = {"search_notes": {"terms"}, "read_note": {"note_id"},
        "evaluate_rule": {"rule_id"}, "finish": set()}

def gate(run, proposal):
    tool = proposal.get("tool")
    args = proposal.get("args", {})
    if tool not in ARGS:
        return f"tool not allowed: {tool}"
    if set(args) != ARGS[tool]:
        return f"bad arguments for {tool}: {sorted(args)}"
    rules = [r.id for r in run.protocol.criteria]
    if tool == "evaluate_rule":
        if args["rule_id"] not in rules:
            return f"no such rule: {args['rule_id']}"
    return None

def context_for(run):
    return {"task": run.request,
            "rules": [r.id for r in run.protocol.criteria],
            "results": {k: v[0]
                        for k, v in run.results.items()},
            "observations": [e for e in run.events
                             if e["kind"] == "observation"],
            "steps_left": (run.budget.steps
                           - run.count("proposal"))}

def step(run, planner, reader):
    proposal = planner(context_for(run))
    run.record("proposal", **proposal)
    refusal = gate(run, proposal)
    if refusal:
        run.record("gate", refused=refusal)
        run.record("observation", error=refusal)
        return
    tool, args = proposal["tool"], proposal.get("args", {})
    if tool == "finish":
        missing = [r.id for r in run.protocol.criteria
                   if r.id not in run.results]
        if missing:
            run.record("observation",
                       error=f"cannot finish: {missing}")
            return
        run.state = "completed"
        run.reason = "every rule has a result"
        return
    if tool == "read_note":
        args = {**args, "reader": reader}
    observation = TOOLS[tool](run, **args)
    run.record("observation", tool=tool, **observation)

## 7. Listing 3.5: the loop and its five endings

In [ ]:
def drive(run, planner, reader=fake_reader):
    while run.state == "running":
        if run.cancel:
            run.state = "cancelled"
            run.reason = "asked to stop"
        elif run.count("proposal") >= run.budget.steps:
            run.state = "blocked"
            run.reason = "step budget spent"
        else:
            try:
                step(run, planner, reader)
            except BudgetExhausted as why:
                run.state, run.reason = "blocked", str(why)
            except Exception as why:      # a tool failed
                run.state = "failed"
                run.reason = f"{type(why).__name__}: {why}"
    run.record("stop", state=run.state, reason=run.reason)
    return run

def run_task(req, planner, reader=fake_reader, budget=None):
    try:
        run = start(req, budget)
    except ContractError as why:
        return {"status": "refused", "reason": str(why)}
    return drive(run, planner, reader)

## 8. Listing 3.6: the stand-in planner, and one inspectable run

Every line below was written by the runtime, not by the planner. That is what makes it evidence.

In [ ]:
SEARCH_TERMS = ["anticoagulant", "warfarin", "apixaban"]

def fake_planner(context):
    # Stand-in for a model choosing the next step. It takes
    # the rules in order and searches before it reads.
    obs = context["observations"]
    for rule_id in context["rules"]:
        if rule_id in context["results"]:
            continue
        if rule_id != "anticoag":
            return {"tool": "evaluate_rule",
                    "args": {"rule_id": rule_id}}
        searches = [e for e in obs
                    if e.get("tool") == "search_notes"]
        if not searches:
            return {"tool": "search_notes",
                    "args": {"terms": SEARCH_TERMS}}
        read = {e["note"] for e in obs
                if e.get("tool") == "read_note"}
        todo = [n for n in searches[-1]["notes"]
                if n not in read]
        if todo:
            return {"tool": "read_note",
                    "args": {"note_id": todo[0]}}
        return {"tool": "evaluate_rule",
                "args": {"rule_id": rule_id}}
    return {"tool": "finish"}

run = run_task(request("P041", 3), fake_planner)
for e in run.events:
    print(e)
print()
print(run.state, "|", run.reason)
for rule, (status, why, ref) in run.results.items():
    print(f"  {rule:9s} {status:8s} {why}  [{ref}]")

The checks below are what the chapter asserts about this run.

In [ ]:
assert run.state == "completed"
assert run.count("proposal") == 7 and run.reads == 2
assert run.results["anticoag"][:2] == ("not met", "present 2026-06-15")
assert "n26" in run.problems and run.problems["n26"] == "statement has no date"
assert run.events[-1]["kind"] == "stop"
print("One inspectable run: as described.")

## 9. Listing 3.7 and Table 3.2: four ways to go wrong, and two more

Expected: every trial ends in a named state with a reason. `loopy` and `eager` are stopped by the step budget; `rogue` is refused once and then completes; the broken reader becomes `failed` with the exception in the reason; a read budget of one becomes `blocked`; a cancelled run is `cancelled`.

In [ ]:
def loopy(context):      # never stops searching
    return {"tool": "search_notes",
            "args": {"terms": ["anything"]}}

def eager(context):      # declares victory at once
    return {"tool": "finish"}
def rogue(context):      # tries a tool it was never given
    if not context["observations"]:
        return {"tool": "write_record",
                "args": {"marker": 5}}
    return fake_planner(context)

def broken_reader(context):
    raise RuntimeError("model endpoint timed out")

req = request("P041", 3)
trials = [("planner", fake_planner, fake_reader, None),
          ("loopy", loopy, fake_reader, None),
          ("eager", eager, fake_reader, None),
          ("rogue", rogue, fake_reader, None),
          ("broken", fake_planner, broken_reader, None),
          ("one read", fake_planner, fake_reader,
           Budget(reads=1))]
for name, planner, reader, budget in trials:
    run = run_task(req, planner, reader, budget)
    print(f"{name:9s} {run.state:10s} {run.reason[:26]:26s}"
          f" steps={run.count('proposal'):2d} "
          f"reads={run.reads}")
run = start(req)
run.cancel = True
drive(run, fake_planner)
print(f"{'cancelled':9s} {run.state:10s} {run.reason}")

In [ ]:
outcomes = {name: run_task(request("P041", 3), pl, rd, bg)
            for name, pl, rd, bg in trials}
assert outcomes["loopy"].state == "blocked" and outcomes["loopy"].count("proposal") == 12
assert outcomes["eager"].state == "blocked"
assert outcomes["rogue"].state == "completed" and outcomes["rogue"].count("gate") == 1
assert outcomes["broken"].state == "failed" and "RuntimeError" in outcomes["broken"].reason
assert outcomes["one read"].state == "blocked" and outcomes["one read"].reason == "read budget spent"
print("Exit test: a planner that repeats a request terminates at the budget, and the log says which one.")
print("rogue's refusal:", next(e for e in outcomes["rogue"].events if e["kind"] == "gate"))

## 10. Exercises

### Exercise 3.1: a planner that cannot make progress

Give `eager` a budget of three steps and read its log. Then change `step` so that a second refused `finish` in a row ends the run as blocked with the reason "planner cannot make progress".

In [ ]:
short = run_task(request("P041", 3), eager, budget=Budget(steps=3))
for e in short.events: print(e)

In [ ]:
_original_step = step

def step(run, planner, reader):
    _original_step(run, planner, reader)
    obs = [e for e in run.events if e["kind"] == "observation"]
    stuck = [e for e in obs[-2:] if str(e.get("error", "")).startswith("cannot finish")]
    if run.state == "running" and len(stuck) == 2:
        run.state, run.reason = "blocked", "planner cannot make progress"

again = run_task(request("P041", 3), eager)
print(again.state, "|", again.reason, "| steps:", again.count("proposal"))
assert again.state == "blocked" and again.count("proposal") == 2
step = _original_step

**Solution 3.1.** With three steps `eager` proposes `finish` three times, is told three times which rules lack results, and is stopped by the budget. Ending after two refused finishes in a row saves ten wasted steps and gives the reviewer a more useful reason than "step budget spent".

When would it be wrong? When a refusal is expected to change the planner's behaviour and it needs one more turn to act on it: a live model that, after being told "no result for anticoag", proposes `search_notes` on its next turn. Two identical refusals in a row is a reasonable signal of no progress; one is not. The general lesson is that "cannot make progress" is a policy the harness owns, and its threshold belongs in `Budget`, not in the planner.

### Exercise 3.2: a `draft_packet` tool

Add a fourth tool that assembles the current results into a packet dictionary, and have the planner call it before `finish`.

In [ ]:
def t_draft(run):
    name = f"{run.protocol.study} v{run.protocol.version}"
    run.packet = {"protocol": name, "criteria": dict(run.results),
                  "problems": dict(run.problems), "eligible": None,
                  "review": "pending"}
    return {"packet": sorted(run.packet["criteria"])}

TOOLS["draft_packet"] = t_draft
ARGS["draft_packet"] = set()

def drafting_planner(context):
    proposal = fake_planner(context)
    drafted = any(e.get("tool") == "draft_packet" for e in context["observations"])
    if proposal["tool"] == "finish" and not drafted:
        return {"tool": "draft_packet", "args": {}}
    return proposal

_step_before_draft = step
def step(run, planner, reader):
    # completion check now also requires a drafted packet
    proposal_count = run.count("proposal")
    _step_before_draft(run, planner, reader)
    if run.state == "completed" and not hasattr(run, "packet"):
        run.state = "running"
        run.events.append({"seq": len(run.events), "kind": "observation",
                           "error": "cannot finish: no packet drafted"})

r = run_task(request("P041", 3), drafting_planner)
print(r.state, "|", r.reason)
print(r.packet)
assert r.state == "completed" and r.packet["criteria"]["anticoag"][0] == "not met"
step = _step_before_draft
del TOOLS["draft_packet"], ARGS["draft_packet"]

**Solution 3.2.** The gate needs exactly two things: the tool's name in `TOOLS` and its argument set (empty) in `ARGS`. Nothing else changes; the gate never inspects what a tool does, only whether it may be called and with what. The completion check now checks two things instead of one: every rule has a result, *and* a packet exists. Notice that `draft_packet` writes to the run, not to any record. The harness still cannot write anywhere external, and that is the difference between it and `write_record`, which the gate still refuses.

### Exercise 3.3

Run Section 11 below with a live model and read its proposals. Then decide which of its choices you would let it keep making unwatched, and which you would want a fixture for first. There is no single answer; a reasonable one is at the end of Section 11.

## 11. Optional: the same runtime with a live planner

Configure a model as in the Chapter 1 notebook. `live_planner` receives exactly the context dictionary the stand-in receives, plus the tool list and argument names, and must return one proposal as JSON. The runtime does not change: the same gate, the same tools, the same budgets, the same five endings.

In [ ]:
import os, urllib.request, urllib.error, json

try:
    from databricks.sdk import WorkspaceClient
    _w = WorkspaceClient()
    os.environ.setdefault("DATABRICKS_HOST", _w.config.host)
    _auth = _w.config.authenticate().get("Authorization", "")
    if _auth.startswith("Bearer "):
        os.environ.setdefault("DATABRICKS_TOKEN", _auth.split(" ", 1)[1])
except Exception:
    pass
LIVE = bool(os.environ.get("DATABRICKS_HOST") and os.environ.get("DATABRICKS_TOKEN"))
ENDPOINT = os.environ.get("MODEL_ENDPOINT", "databricks-gpt-5-6-luna")

def chat_json(prompt, fallback):
    host = os.environ["DATABRICKS_HOST"].rstrip("/")
    body = json.dumps({"model": ENDPOINT, "messages": [{"role": "user", "content": prompt}]}).encode()
    req = urllib.request.Request(host + "/serving-endpoints/chat/completions", body,
                                 {"Content-Type": "application/json",
                                  "Authorization": "Bearer " + os.environ["DATABRICKS_TOKEN"]})
    with urllib.request.urlopen(req, timeout=60) as reply:
        text = json.load(reply)["choices"][0]["message"]["content"]
    try:
        return json.loads(text.strip().strip("`").removeprefix("json").strip())
    except ValueError:
        return fallback(text)

def live_planner(context):
    prompt = (
        "You are the planner inside a clinical-trial screening harness. Choose the single next "
        "step. Reply with JSON only: {\"tool\": <name>, \"args\": {...}}.\n"
        "Tools and their exact argument names:\n"
        "  search_notes {\"terms\": [list of words]} -> note ids whose text contains any term\n"
        "  read_note {\"note_id\": id} -> a grounded statement about anticoagulant use, or a problem\n"
        "  evaluate_rule {\"rule_id\": id} -> status for one rule using evidence gathered so far\n"
        "  finish {} -> completion check; refused if any rule has no result\n"
        "Rules that need a result: " + json.dumps(context["rules"]) + "\n"
        "Results so far: " + json.dumps(context["results"]) + "\n"
        "Steps left: " + str(context["steps_left"]) + "\n"
        "Observations so far (oldest first):\n" + json.dumps(context["observations"], default=str) + "\n"
        "Only 'anticoag' depends on notes; evaluate it after reading the relevant notes. Do not re-read a note."
    )
    return chat_json(prompt, lambda text: {"tool": "unparseable", "args": {"text": text[:80]}})

print("Live model available:", LIVE, "| endpoint:", ENDPOINT if LIVE else "(none)")

In [ ]:
import datetime
if LIVE:
    print(f"{datetime.date.today()}  endpoint={ENDPOINT}")
    live = run_task(request("P041", 3), live_planner, budget=Budget(steps=10, reads=4))
    for e in live.events:
        print(e)
    print()
    print(live.state, "|", live.reason, "| steps:", live.count("proposal"), "| reads:", live.reads,
          "| gate refusals:", live.count("gate"))
    for rule, (status, why, ref) in live.results.items():
        print(f"  {rule:9s} {status:8s} {why}  [{ref}]")
else:
    print("Skipped: no live model configured.")

### What to look for, and a reasonable answer to Exercise 3.3

Compare the live log with the stand-in's. Things that commonly differ: the live planner may evaluate the numeric rules in a different order, may search with different terms (and so find more or fewer notes), may try to read a note twice, or may return a proposal the gate refuses (a wrong argument name is the usual one). Every one of those shows up in the log as a proposal, a gate event, or an observation, and the run still ends in one of the five states.

For Exercise 3.3: the choices safe to leave unwatched are the ones the gate and the budgets already bound completely, such as which rule to evaluate next and how many steps to take. The choice that wants a fixture first is the search terms: a planner that searches for "warfarin" but not "apixaban" will miss n25, evaluate the exclusion as unknown, and complete honestly with the wrong answer. Nothing in the runtime catches that; only a fixture with the expected result does, which is what Chapter 9 is for. Print endpoint and date with anything you quote.

## 12. Done

A runtime under two hundred lines, every run a readable log, five named endings, and a planner that can be a stand-in or a model without the runtime knowing. Chapter 4 decides what goes into the context dictionary and what stays out.

In [ ]:
print("Chapter 3 complete.")

## References

- [Yao et al., ReAct](https://arxiv.org/abs/2210.03629): the reason, act, observe cycle.
- [Pi Agent Harness](https://github.com/earendil-works/pi): separate agent-core and model-adapter packages.
- [mini-swe-agent](https://github.com/SWE-agent/mini-swe-agent): a deliberately small loop.

The runtime, notes, and exercises are original companion material.